# C09 Final reseed comparison

Aggregate the top-3 reseed runs, compare stability, and produce a final recommendation table for the thesis.


In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


In [3]:
CWD = Path.cwd()
NOTEBOOKS_DIR = CWD.parent if CWD.name == "challengers" else CWD
PROJECT_ROOT = NOTEBOOKS_DIR.parent
FIGURES_DIR = PROJECT_ROOT / "figures" / "challengers"
MODELS_DIR = PROJECT_ROOT / "notebooks" / "models" / "challengers"

RUNS_CSV = FIGURES_DIR / "c08_top3_reseed_runs.csv"
FINAL_TABLE_CSV = FIGURES_DIR / "c09_final_reseed_comparison.csv"
FINAL_DECISION_JSON = FIGURES_DIR / "c09_final_model_recommendation.json"
FINAL_PLOT = FIGURES_DIR / "c09_reseed_tradeoff_scatter.png"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RUNS_CSV exists: {RUNS_CSV.exists()}")


PROJECT_ROOT: /Users/natashaagapova/Documents/A-INNOPOLIS/A-THESIS/my-repository
RUNS_CSV exists: False


In [4]:
def _safe_float(value):
    if value is None:
        return np.nan
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan


if RUNS_CSV.exists():
    runs_df = pd.read_csv(RUNS_CSV)
else:
    rows = []
    for config_path in sorted(MODELS_DIR.glob("*_seed*/training_config.json")):
        config = json.loads(config_path.read_text())
        rows.append({
            "model_name": config_path.parent.name,
            "base_model_name": config.get("base_model_name"),
            "method": config.get("method"),
            "recipe": config.get("recipe"),
            "seed": config.get("seed"),
            "epochs": config.get("epochs"),
            "accuracy": _safe_float(config.get("accuracy")),
            "macro_f1": _safe_float(config.get("macro_f1")),
            "worst_gap": _safe_float(config.get("tpr_gap_worst_robust")),
            "macro_gap": _safe_float(config.get("tpr_gap_macro_robust")),
            "model_dir": str(config_path.parent.relative_to(PROJECT_ROOT)),
        })
    runs_df = pd.DataFrame(rows)

if runs_df.empty:
    raise FileNotFoundError("No reseed runs found. Run c08_top3_reseed.ipynb first.")

runs_df = runs_df.sort_values(["base_model_name", "seed"]).reset_index(drop=True)
runs_df


FileNotFoundError: No reseed runs found. Run c08_top3_reseed.ipynb first.

In [ ]:
agg_df = runs_df.groupby(["base_model_name", "method", "recipe"], dropna=False).agg(
    runs=("seed", "count"),
    seeds=("seed", lambda s: ",".join(str(int(x)) for x in sorted(s))),
    accuracy_mean=("accuracy", "mean"),
    accuracy_std=("accuracy", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    worst_gap_mean=("worst_gap", "mean"),
    worst_gap_std=("worst_gap", "std"),
    macro_gap_mean=("macro_gap", "mean"),
    macro_gap_std=("macro_gap", "std"),
).reset_index()

agg_df = agg_df.sort_values(["macro_f1_mean", "worst_gap_mean"], ascending=[False, True]).reset_index(drop=True)
agg_df.insert(0, "final_rank", np.arange(1, len(agg_df) + 1))
agg_df


In [ ]:
leader = agg_df.iloc[0].to_dict()
decision = {
    "recommended_model": leader["base_model_name"],
    "method": leader["method"],
    "runs": int(leader["runs"]),
    "seeds": leader["seeds"],
    "macro_f1_mean": float(leader["macro_f1_mean"]),
    "macro_f1_std": float(leader["macro_f1_std"]) if pd.notna(leader["macro_f1_std"]) else None,
    "worst_gap_mean": float(leader["worst_gap_mean"]),
    "macro_gap_mean": float(leader["macro_gap_mean"]),
}

agg_df.to_csv(FINAL_TABLE_CSV, index=False)
FINAL_DECISION_JSON.write_text(json.dumps(decision, indent=2))

print(f"Saved final table to: {FINAL_TABLE_CSV}")
print(f"Saved final decision to: {FINAL_DECISION_JSON}")
agg_df


In [ ]:
display_cols = [
    "final_rank",
    "base_model_name",
    "method",
    "runs",
    "seeds",
    "accuracy_mean",
    "accuracy_std",
    "macro_f1_mean",
    "macro_f1_std",
    "worst_gap_mean",
    "macro_gap_mean",
]
agg_df[display_cols]


In [ ]:
if plt is None:
    print("matplotlib is not installed; skipping plots")
else:
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.errorbar(
        agg_df["worst_gap_mean"],
        agg_df["macro_f1_mean"],
        xerr=agg_df["worst_gap_std"].fillna(0.0),
        yerr=agg_df["macro_f1_std"].fillna(0.0),
        fmt="o",
        capsize=4,
    )
    for _, row in agg_df.iterrows():
        ax.annotate(row["base_model_name"], (row["worst_gap_mean"], row["macro_f1_mean"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
    ax.set_xlabel("Mean worst TPR gap (lower is better)")
    ax.set_ylabel("Mean macro-F1 (higher is better)")
    ax.set_title("Top-3 reseed comparison")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(FINAL_PLOT, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved plot to: {FINAL_PLOT}")


In [ ]:
print("Recommended final model:")
print(json.dumps(decision, indent=2))
